### Texte Einlesen usw.

In [11]:
import os
import json

BASE_DIR = os.getcwd()
print("Arbeitsverzeichnis:", BASE_DIR)

CATEGORIES = {
    "moderne Mythologien": "Mythologien",
    "Orbitalsysteme": "Orbitalsysteme"
}

def collect_data(category_folder, audio_id):
    data = []
    experiment_id = 1

    category_path = os.path.join(BASE_DIR, category_folder)

    if not os.path.isdir(category_path):
        print(f"⚠ Ordner nicht gefunden: {category_path}")
        return data

    for model_name in sorted(os.listdir(category_path)):
        model_path = os.path.join(category_path, model_name)

        if not os.path.isdir(model_path):
            continue

        run_id = 1
        for filename in sorted(os.listdir(model_path)):
            if filename.endswith(".txt"):
                file_path = os.path.join(model_path, filename)

                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read().strip()

                entry = {
                    "experiment_id": experiment_id,
                    "audio_id": audio_id,
                    "model_name": model_name,
                    "run_id": run_id,
                    "text": text
                }

                data.append(entry)

                experiment_id += 1
                run_id += 1

    return data


# NDJSON schreiben
for folder, audio_id in CATEGORIES.items():
    output_filename = audio_id.lower() + ".jsonl"
    output_path = os.path.join(BASE_DIR, output_filename)

    items = collect_data(folder, audio_id)

    with open(output_path, "w", encoding="utf-8") as f:
        for entry in items:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"✔ NDJSON-Datei erzeugt: {output_filename} ({len(items)} Einträge)")


Arbeitsverzeichnis: /home/wattaah/embedding/old/Gegenexperiment
✔ NDJSON-Datei erzeugt: mythologien.jsonl (16 Einträge)
✔ NDJSON-Datei erzeugt: orbitalsysteme.jsonl (16 Einträge)


In [12]:
import os
import json

BASE_DIR = os.getcwd()

# Zuordnung Datei → audio_id
TRANSKRIPTE = {
    "moderne Mythologien/moderne Mythologien.txt": "Mythologien",
    "Orbitalsysteme/Orbitalsysteme.txt": "Orbitalsysteme"
}

output_file = os.path.join(BASE_DIR, "transkripte.jsonl")

with open(output_file, "w", encoding="utf-8") as out:
    for rel_path, audio_id in TRANSKRIPTE.items():
        file_path = os.path.join(BASE_DIR, rel_path)

        if not os.path.isfile(file_path):
            print(f"⚠ Datei nicht gefunden: {file_path}")
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        entry = {
            "audio_id": audio_id,
            "text": text
        }

        out.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("✔ NDJSON-Datei erstellt:", output_file)


✔ NDJSON-Datei erstellt: /home/wattaah/embedding/old/Gegenexperiment/transkripte.jsonl


### Embedding Llama Modell 

In [13]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# 🔥 Dein lokaler Llama-Client
client = OpenAI(
    base_url="http://localhost:8002/v1",
    api_key="none"
)

# -----------------------------
# 1️⃣ Dateien einlesen
# -----------------------------
df_myth = pd.read_json("mythologien.jsonl", lines=True)
df_orbit = pd.read_json("orbitalsysteme.jsonl", lines=True)

print("Mythologien:", len(df_myth))
print("Orbitalsysteme:", len(df_orbit))

# -----------------------------
# 2️⃣ Embedding-Helper
# -----------------------------
def compute_embeddings(df, model_name="meta-llama/Llama-3.1-8B-Instruct"):
    embeddings = []

    for text in tqdm(df["text"], desc=f"Embeddings für {model_name}"):
        if not isinstance(text, str) or not text.strip():
            embeddings.append(None)
            continue
        
        # 🔥 Embedding über deinen lokalen Server
        response = client.embeddings.create(
            model=model_name,
            input=text
        )
        emb = response.data[0].embedding
        embeddings.append(emb)

    df[f"embedding_{model_name.replace('/', '_')}"] = embeddings
    return df


# -----------------------------
# 3️⃣ Embeddings für beide DataFrames
# -----------------------------
model = "meta-llama/Llama-3.1-8B-Instruct"

df_myth_emb = compute_embeddings(df_myth.copy(), model)
df_orbit_emb = compute_embeddings(df_orbit.copy(), model)

# -----------------------------
# 4️⃣ Kombinierte Version
# -----------------------------
df_combined = pd.concat([df_myth_emb, df_orbit_emb], ignore_index=True)

# -----------------------------
# 5️⃣ Alles speichern
# -----------------------------
df_myth_emb.to_parquet("mythologien_with_embeddings.parquet")
df_orbit_emb.to_parquet("orbitalsysteme_with_embeddings.parquet")
df_combined.to_parquet("combined_llm_embeddings.parquet")

print("\n✅ Fertig! Drei Dateien gespeichert:")
print(" - mythologien_with_embeddings.parquet")
print(" - orbitalsysteme_with_embeddings.parquet")
print(" - combined_llm_embeddings.parquet")


Mythologien: 16
Orbitalsysteme: 16


Embeddings für meta-llama/Llama-3.1-8B-Instruct: 100%|██████████| 16/16 [00:26<00:00,  1.67s/it]


✅ Fertig! Drei Dateien gespeichert:
 - mythologien_with_embeddings.parquet
 - orbitalsysteme_with_embeddings.parquet
 - combined_llm_embeddings.parquet


### Embedding mit Jina

In [15]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --------------------------------------------------
# 1️⃣ Liste der Parquet-Dateien, für die Embeddings berechnet werden sollen
# --------------------------------------------------
parquet_files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet"
]

# --------------------------------------------------
# 2️⃣ JinaAI Embedding-Modell laden
# --------------------------------------------------
model_name = "jinaai/jina-embeddings-v3"
col_name = "embedding_Jina_v3"

print(f"🔄 Lade JinaAI Embedding Modell: {model_name}")
model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device="cuda"
)

# --------------------------------------------------
# 3️⃣ Funktion, die Jina-Embeddings zu einer Datei hinzufügt
# --------------------------------------------------
def add_jina_embeddings(parquet_path):

    print(f"\n📄 Lade Datei: {parquet_path}")
    df = pd.read_parquet(parquet_path)

    # Falls die Spalte bereits existiert → Skip
    if col_name in df.columns:
        print(f"⚠️  Spalte '{col_name}' existiert bereits → Überspringe.")
        return

    # Texte vorbereiten
    texts = df["text"].astype(str).tolist()

    print(f"➡️  Berechne Jina-Embeddings für {len(df)} Texte …")

    # Embeddings berechnen
    embeddings = model.encode(
        texts,
        batch_size=2,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    # In DataFrame speichern
    df[col_name] = embeddings.tolist()

    # Datei überschreiben
    df.to_parquet(parquet_path)
    print(f"💾 Gespeichert: {parquet_path} (inkl. Spalte '{col_name}')")


# --------------------------------------------------
# 4️⃣ Schleife über ALLE 3 Dateien
# --------------------------------------------------
for file in parquet_files:
    add_jina_embeddings(file)

print("\n✅ JinaAI Embeddings wurden erfolgreich in ALLE drei Dateien eingefügt!")


🔄 Lade JinaAI Embedding Modell: jinaai/jina-embeddings-v3


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention impl


📄 Lade Datei: mythologien_with_embeddings.parquet
➡️  Berechne Jina-Embeddings für 16 Texte …


Batches: 100%|██████████| 8/8 [00:00<00:00, 13.02it/s]


💾 Gespeichert: mythologien_with_embeddings.parquet (inkl. Spalte 'embedding_Jina_v3')

📄 Lade Datei: orbitalsysteme_with_embeddings.parquet
➡️  Berechne Jina-Embeddings für 16 Texte …


Batches: 100%|██████████| 8/8 [00:00<00:00, 31.57it/s]


💾 Gespeichert: orbitalsysteme_with_embeddings.parquet (inkl. Spalte 'embedding_Jina_v3')

📄 Lade Datei: combined_llm_embeddings.parquet
➡️  Berechne Jina-Embeddings für 32 Texte …


Batches: 100%|██████████| 16/16 [00:00<00:00, 43.94it/s]

💾 Gespeichert: combined_llm_embeddings.parquet (inkl. Spalte 'embedding_Jina_v3')

✅ JinaAI Embeddings wurden erfolgreich in ALLE drei Dateien eingefügt!


## Embedding mit aari 

In [16]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --------------------------------------------------
# 1️⃣ Liste deiner Parquet-Dateien
# --------------------------------------------------
parquet_files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet"
]

# --------------------------------------------------
# 2️⃣ AARI Embedding-Modell laden
# --------------------------------------------------
model_name = "aari1995/German_Semantic_V3b"
col_name = "embedding_AARI_German_Semantic_V3b"

print(f"🔄 Lade AARI Embedding Modell: {model_name}")
model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device="cpu"   # CPU, weil CUDA OOM
)

# --------------------------------------------------
# 3️⃣ Embedding-Funktion
# --------------------------------------------------
def add_aari_embeddings(parquet_path):

    print(f"\n📄 Lade Datei: {parquet_path}")
    df = pd.read_parquet(parquet_path)

    # Falls Spalte bereits existiert → Skip
    if col_name in df.columns:
        print(f"⚠️  Spalte '{col_name}' existiert bereits → Überspringe.")
        return

    print(f"➡️  Berechne AARI-Embeddings für {len(df)} Texte …")

    texts = df["text"].astype(str).tolist()

    embeddings = model.encode(
        texts,
        batch_size=1,                # AARI ist schwerer → kleine Batches
        normalize_embeddings=True,
        show_progress_bar=True
    )

    df[col_name] = embeddings.tolist()

    # Datei überschreiben
    df.to_parquet(parquet_path)
    print(f"💾 Gespeichert: {parquet_path} (inkl. '{col_name}')")


# --------------------------------------------------
# 4️⃣ Schleife über alle Dateien
# --------------------------------------------------
for file in parquet_files:
    add_aari_embeddings(file)

print("\n✅ AARI-Embeddings erfolgreich in ALLE drei Dateien eingefügt!")


🔄 Lade AARI Embedding Modell: aari1995/German_Semantic_V3b

📄 Lade Datei: mythologien_with_embeddings.parquet
➡️  Berechne AARI-Embeddings für 16 Texte …


Batches: 100%|██████████| 16/16 [00:07<00:00,  2.01it/s]


💾 Gespeichert: mythologien_with_embeddings.parquet (inkl. 'embedding_AARI_German_Semantic_V3b')

📄 Lade Datei: orbitalsysteme_with_embeddings.parquet
➡️  Berechne AARI-Embeddings für 16 Texte …


Batches: 100%|██████████| 16/16 [00:10<00:00,  1.60it/s]


💾 Gespeichert: orbitalsysteme_with_embeddings.parquet (inkl. 'embedding_AARI_German_Semantic_V3b')

📄 Lade Datei: combined_llm_embeddings.parquet
➡️  Berechne AARI-Embeddings für 32 Texte …


Batches: 100%|██████████| 32/32 [00:16<00:00,  1.97it/s]

💾 Gespeichert: combined_llm_embeddings.parquet (inkl. 'embedding_AARI_German_Semantic_V3b')

✅ AARI-Embeddings erfolgreich in ALLE drei Dateien eingefügt!


## Embedding mit BAAI

In [17]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --------------------------------------------------
# 1️⃣ Liste aller Parquet-Dateien
# --------------------------------------------------
parquet_files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet"
]

# --------------------------------------------------
# 2️⃣ BGE Embedding Modell laden
# --------------------------------------------------
model_name = "BAAI/bge-m3"
col_name = "embedding_BGE_m3"

print(f"🔄 Lade Embedding Modell: {model_name}")

model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device="cuda"     # Falls OOM → device="cpu"
)

# --------------------------------------------------
# 3️⃣ Funktion: Embeddings zu einer Datei hinzufügen
# --------------------------------------------------
def add_bge_embeddings(parquet_path):

    print(f"\n📄 Lade Datei: {parquet_path}")
    df = pd.read_parquet(parquet_path)

    # Falls Spalte schon existiert → Skip
    if col_name in df.columns:
        print(f"⚠️  Spalte '{col_name}' existiert bereits → Überspringe.")
        return

    texts = df["text"].astype(str).tolist()

    print(f"➡️ Berechne BGE-m3 Embeddings für {len(df)} Texte …")

    embeddings = model.encode(
        texts,
        batch_size=1,               # BGE ist speicherintensiv
        normalize_embeddings=True,
        show_progress_bar=True
    )

    df[col_name] = embeddings.tolist()

    df.to_parquet(parquet_path)
    print(f"💾 Gespeichert: {parquet_path} (inkl. '{col_name}')")


# --------------------------------------------------
# 4️⃣ Schleife über alle Dateien
# --------------------------------------------------
for file in parquet_files:
    add_bge_embeddings(file)

print("\n✅ BGE-m3 Embeddings erfolgreich in ALLE drei Dateien eingefügt!")


🔄 Lade Embedding Modell: BAAI/bge-m3

📄 Lade Datei: mythologien_with_embeddings.parquet
➡️ Berechne BGE-m3 Embeddings für 16 Texte …


Batches: 100%|██████████| 16/16 [00:00<00:00, 34.83it/s]


💾 Gespeichert: mythologien_with_embeddings.parquet (inkl. 'embedding_BGE_m3')

📄 Lade Datei: orbitalsysteme_with_embeddings.parquet
➡️ Berechne BGE-m3 Embeddings für 16 Texte …


Batches: 100%|██████████| 16/16 [00:00<00:00, 37.92it/s]


💾 Gespeichert: orbitalsysteme_with_embeddings.parquet (inkl. 'embedding_BGE_m3')

📄 Lade Datei: combined_llm_embeddings.parquet
➡️ Berechne BGE-m3 Embeddings für 32 Texte …


Batches: 100%|██████████| 32/32 [00:00<00:00, 40.73it/s]

💾 Gespeichert: combined_llm_embeddings.parquet (inkl. 'embedding_BGE_m3')

✅ BGE-m3 Embeddings erfolgreich in ALLE drei Dateien eingefügt!


### Embeddding mit Alibaba

In [18]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --------------------------------------------------
# 1️⃣ Liste deiner Parquet-Dateien
# --------------------------------------------------
parquet_files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet"
]

# --------------------------------------------------
# 2️⃣ Alibaba Embedding Modell laden
# --------------------------------------------------
model_name = "Alibaba-NLP/gte-multilingual-base"
col_name = "embedding_Alibaba_gte_multilingual_base"

print(f"🔄 Lade Embedding Modell: {model_name}")

model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device="cuda"     # Falls OOM → device="cpu"
)

# --------------------------------------------------
# 3️⃣ Funktion: Embeddings zu einer Datei hinzufügen
# --------------------------------------------------
def add_alibaba_embeddings(parquet_path):

    print(f"\n📄 Lade Datei: {parquet_path}")
    df = pd.read_parquet(parquet_path)

    # Falls die Spalte schon existiert → überspringen
    if col_name in df.columns:
        print(f"⚠️  Spalte '{col_name}' existiert bereits → Überspringe.")
        return

    texts = df["text"].astype(str).tolist()

    print(f"➡️ Berechne Alibaba GTE Embeddings für {len(df)} Texte …")

    embeddings = model.encode(
        texts,
        batch_size=1,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    df[col_name] = embeddings.tolist()

    df.to_parquet(parquet_path)
    print(f"💾 Gespeichert: {parquet_path} (inkl. '{col_name}')")


# --------------------------------------------------
# 4️⃣ Schleife über ALLE 3 Dateien
# --------------------------------------------------
for file in parquet_files:
    add_alibaba_embeddings(file)

print("\n✅ Alibaba GTE Embeddings erfolgreich in ALLE drei Dateien eingefügt!")


🔄 Lade Embedding Modell: Alibaba-NLP/gte-multilingual-base


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



📄 Lade Datei: mythologien_with_embeddings.parquet
➡️ Berechne Alibaba GTE Embeddings für 16 Texte …


Batches: 100%|██████████| 16/16 [00:00<00:00, 91.86it/s]


💾 Gespeichert: mythologien_with_embeddings.parquet (inkl. 'embedding_Alibaba_gte_multilingual_base')

📄 Lade Datei: orbitalsysteme_with_embeddings.parquet
➡️ Berechne Alibaba GTE Embeddings für 16 Texte …


Batches: 100%|██████████| 16/16 [00:00<00:00, 78.88it/s]


💾 Gespeichert: orbitalsysteme_with_embeddings.parquet (inkl. 'embedding_Alibaba_gte_multilingual_base')

📄 Lade Datei: combined_llm_embeddings.parquet
➡️ Berechne Alibaba GTE Embeddings für 32 Texte …


Batches: 100%|██████████| 32/32 [00:00<00:00, 87.29it/s]

💾 Gespeichert: combined_llm_embeddings.parquet (inkl. 'embedding_Alibaba_gte_multilingual_base')

✅ Alibaba GTE Embeddings erfolgreich in ALLE drei Dateien eingefügt!


### der Inhalt der df anzeigen 

In [19]:
import pandas as pd

# Liste deiner drei Dateien
files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet"
]

for file in files:
    print(f"\n📄 Lade Datei: {file}")
    df = pd.read_parquet(file)

    print("✅ Datei geladen!")
    print(f"Anzahl Zeilen: {len(df)}")
    print("Spalten:")
    print(list(df.columns))
    print("-" * 60)



📄 Lade Datei: mythologien_with_embeddings.parquet
✅ Datei geladen!
Anzahl Zeilen: 16
Spalten:
['experiment_id', 'audio_id', 'model_name', 'run_id', 'text', 'embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base']
------------------------------------------------------------

📄 Lade Datei: orbitalsysteme_with_embeddings.parquet
✅ Datei geladen!
Anzahl Zeilen: 16
Spalten:
['experiment_id', 'audio_id', 'model_name', 'run_id', 'text', 'embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base']
------------------------------------------------------------

📄 Lade Datei: combined_llm_embeddings.parquet
✅ Datei geladen!
Anzahl Zeilen: 32
Spalten:
['experiment_id', 'audio_id', 'model_name', 'run_id', 'text', 'embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embe

## jetzt noch die Transkripte einbetten 

In [21]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# --------------------------------------------------
# 1️⃣ Transkripte laden
# --------------------------------------------------
df = pd.read_json("transkripte.jsonl", lines=True)
print("Geladen:", len(df), "Transkripte\n")

texts = df["text"].astype(str).tolist()

# --------------------------------------------------
# 2️⃣ Alle Embedding-Modelle definieren
# --------------------------------------------------
embedding_models = {
    "embedding_Jina_v3":            "jinaai/jina-embeddings-v3",
    "embedding_AARI_German_V3b":    "aari1995/German_Semantic_V3b",
    "embedding_BGE_m3":             "BAAI/bge-m3",
    "embedding_Alibaba_GTE":        "Alibaba-NLP/gte-multilingual-base",
}

# Falls manche Modelle nicht gewünscht oder nicht installiert → einfach auskommentieren
# del embedding_models["embedding_Llama_8B_Instruct"]

# --------------------------------------------------
# 3️⃣ Embeddings für jedes Modell berechnen
# --------------------------------------------------
for col_name, model_name in embedding_models.items():

    print(f"\n🔄 Lade Modell: {model_name}")
    try:
        model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device="cuda"   # Bei Fehler wegen OOM → "cpu"
        )
    except Exception as e:
        print(f"⚠️ Fehler beim Laden von {model_name}: {e}")
        print("➡️ Benutze CPU stattdessen.")
        model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device="cpu"
        )

    print(f"➡️ Berechne Embeddings für: {col_name}")

    embeddings = model.encode(
        texts,
        batch_size=1,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    df[col_name] = embeddings.tolist()

# --------------------------------------------------
# 4️⃣ Speichern
# --------------------------------------------------
df.to_parquet("transkripte_with_embeddings.parquet")

print("\n✅ Alle Embeddings gespeichert in: transkripte_with_embeddings.parquet")


Geladen: 2 Transkripte


🔄 Lade Modell: jinaai/jina-embeddings-v3


flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i

➡️ Berechne Embeddings für: embedding_Jina_v3


Batches: 100%|██████████| 2/2 [00:00<00:00, 36.76it/s]


🔄 Lade Modell: aari1995/German_Semantic_V3b


...


🔄 Lade Modell: Alibaba-NLP/gte-multilingual-base



Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


➡️ Berechne Embeddings für: embedding_Alibaba_GTE


Batches: 100%|██████████| 2/2 [00:00<00:00, 59.93it/s]


✅ Alle Embeddings gespeichert in: transkripte_with_embeddings.parquet


### Dazu noch Llama Embedding für die 2 Texte 

In [22]:
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# 🔥 Dein lokaler LLaMA-Client
client = OpenAI(
    base_url="http://localhost:8002/v1",
    api_key="none"
)

# -----------------------------
# 1️⃣ transkripte_with_embeddings.parquet einlesen
# -----------------------------
df = pd.read_parquet("transkripte_with_embeddings.parquet")

print("Geladene Transkripte:", len(df))
print("Spalten:", df.columns.tolist())

# -----------------------------
# 2️⃣ LLaMA Embedding-Funktion
# -----------------------------
def compute_llama_embeddings(df, model_name="meta-llama/Llama-3.1-8B-Instruct"):
    col_name = f"embedding_{model_name.replace('/', '_')}"
    
    # Falls bereits vorhanden → nicht neu berechnen
    if col_name in df.columns:
        print(f"⚠️ Spalte '{col_name}' existiert bereits → überspringe Berechnung.")
        return df
    
    embeddings = []

    for text in tqdm(df["text"], desc=f"Berechne {col_name}"):
        if not isinstance(text, str) or not text.strip():
            embeddings.append(None)
            continue
        
        # 🔥 Embedding über lokalen Server erzeugen
        response = client.embeddings.create(
            model=model_name,
            input=text
        )
        emb = response.data[0].embedding
        embeddings.append(emb)

    df[col_name] = embeddings
    return df


# -----------------------------
# 3️⃣ LLaMA Embeddings erzeugen
# -----------------------------
model = "meta-llama/Llama-3.1-8B-Instruct"
df = compute_llama_embeddings(df, model)

# -----------------------------
# 4️⃣ Datei speichern
# -----------------------------
df.to_parquet("transkripte_with_embeddings.parquet")

print("\n✅ Fertig! LLaMA-Embeddings zu 'transkripte_with_embeddings.parquet' hinzugefügt.")


Geladene Transkripte: 2
Spalten: ['audio_id', 'text', 'embedding_Jina_v3', 'embedding_AARI_German_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_GTE']


Berechne embedding_meta-llama_Llama-3.1-8B-Instruct: 100%|██████████| 2/2 [00:04<00:00,  2.41s/it]


✅ Fertig! LLaMA-Embeddings zu 'transkripte_with_embeddings.parquet' hinzugefügt.


### Alles anzeigen 

In [24]:
import pandas as pd

files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet",
    "transkripte_with_embeddings.parquet"
]

for file in files:
    print(f"\n📄 Lade Datei: {file}")
    df = pd.read_parquet(file)

    # Sicherstellen, dass text existiert
    if "text" not in df.columns:
        print(f"⚠️ Datei {file} enthält keine Spalte 'text' – übersprungen!")
        continue

    print(f"🔧 Entferne \\n aus {len(df)} Texten ...")

    # \n und \r entfernen
    df["text"] = (
        df["text"]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
    )

    # Optional: Mehrfache Leerzeichen reduzieren
    df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()

    df.to_parquet(file)
    print(f"💾 Gespeichert: {file}")

print("\n✅ Alle Dateien wurden bereinigt und gespeichert!")



📄 Lade Datei: mythologien_with_embeddings.parquet
🔧 Entferne \n aus 16 Texten ...
💾 Gespeichert: mythologien_with_embeddings.parquet

📄 Lade Datei: orbitalsysteme_with_embeddings.parquet
🔧 Entferne \n aus 16 Texten ...
💾 Gespeichert: orbitalsysteme_with_embeddings.parquet

📄 Lade Datei: combined_llm_embeddings.parquet
🔧 Entferne \n aus 32 Texten ...
💾 Gespeichert: combined_llm_embeddings.parquet

📄 Lade Datei: transkripte_with_embeddings.parquet
🔧 Entferne \n aus 2 Texten ...
💾 Gespeichert: transkripte_with_embeddings.parquet

✅ Alle Dateien wurden bereinigt und gespeichert!


In [2]:
import pandas as pd

# Liste deiner 4 Dateien
files = [
    "mythologien_with_embeddings.parquet",
    "orbitalsysteme_with_embeddings.parquet",
    "combined_llm_embeddings.parquet",
    "transkripte_with_embeddings.parquet"
]

for file in files:
    print("\n" + "="*80)
    print(f"📄 Datei: {file}")

    df = pd.read_parquet(file)

    print(f"➡️ Anzahl Zeilen: {len(df)}")
    print("➡️ Spalten:", df.columns.tolist())

    print("\n🔍 df.head():")
    display(df.head())   # funktioniert in Jupyter optimal

print("\n✅ Alle Dateien erfolgreich angezeigt!")



📄 Datei: mythologien_with_embeddings.parquet
➡️ Anzahl Zeilen: 16
➡️ Spalten: ['experiment_id', 'audio_id', 'model_name', 'run_id', 'text', 'embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base']

🔍 df.head():


,experiment_id,audio_id,model_name,run_id,text,embedding_meta-llama_Llama-3.1-8B-Instruct,embedding_Jina_v3,embedding_AARI_German_Semantic_V3b,embedding_BGE_m3,embedding_Alibaba_gte_multilingual_base
0,1,Mythologien,GPT OSS,1,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.006361679174005985, -0.017968954518437386,...","[0.201171875, -0.1220703125, 0.1630859375, -0....","[0.04692911356687546, -0.07531177252531052, 0....","[-0.03595797345042229, 0.003199505852535367, -...","[-0.022312983870506287, 0.014333673752844334, ..."
1,2,Mythologien,GPT OSS,2,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.004676091484725475, -0.018816368654370308,...","[0.197265625, -0.11865234375, 0.1552734375, -0...","[0.04166073724627495, -0.07697830349206924, 0....","[-0.021111775189638138, 0.00964306015521288, -...","[-0.02722461149096489, 0.010483560152351856, -..."
2,3,Mythologien,GPT OSS,3,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.007357350084930658, -0.015717975795269012,...","[0.2021484375, -0.1181640625, 0.1494140625, -0...","[0.043072812259197235, -0.0732586681842804, 0....","[-0.016145789995789528, 0.010927277617156506, ...","[-0.0174740981310606, 0.00829455815255642, -0...."
3,4,Mythologien,GPT OSS,4,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.006447131745517254, -0.020988065749406815,...","[0.205078125, -0.107421875, 0.1572265625, -0.0...","[0.04661232605576515, -0.06668359786272049, 0....","[-0.03934884071350098, 0.008379417471587658, 0...","[-0.02418340928852558, 0.005975430831313133, -..."
4,5,Mythologien,Llama 3.1,1,### Zusammenfassung: Die Moderne behauptet oft...,"[-0.009736663661897182, -0.014852538704872131,...","[0.17578125, -0.08251953125, 0.1337890625, -0....","[0.056146636605262756, -0.07163380086421967, 0...","[-0.00936897937208414, 0.01469916570931673, -0...","[-0.07917217165231705, 0.03245813772082329, -0..."



📄 Datei: orbitalsysteme_with_embeddings.parquet
➡️ Anzahl Zeilen: 16
➡️ Spalten: ['experiment_id', 'audio_id', 'model_name', 'run_id', 'text', 'embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base']

🔍 df.head():


,experiment_id,audio_id,model_name,run_id,text,embedding_meta-llama_Llama-3.1-8B-Instruct,embedding_Jina_v3,embedding_AARI_German_Semantic_V3b,embedding_BGE_m3,embedding_Alibaba_gte_multilingual_base
0,1,Orbitalsysteme,GPT OSS,1,**Zusammenfassung** Die Raumfahrt des 21. Jahr...,"[-0.005311646964401007, -0.017536865547299385,...","[0.1669921875, -0.0439453125, 0.076171875, -0....","[0.030519453808665276, -0.038564734160900116, ...","[-0.017560848966240883, 0.03174842894077301, -...","[0.028798948973417282, -0.005140561144798994, ..."
1,2,Orbitalsysteme,GPT OSS,2,**Zusammenfassung** Die Raumfahrt des 21. Jahr...,"[-0.01047920249402523, -0.016387687996029854, ...","[0.1630859375, -0.011474609375, 0.08447265625,...","[0.04032159969210625, -0.03397618979215622, 0....","[-0.04326508566737175, 0.02692524343729019, -0...","[0.007778689730912447, 0.005986569449305534, -..."
2,3,Orbitalsysteme,GPT OSS,3,**Zusammenfassung** Die Raumfahrt des 21. Jahr...,"[-0.008527858182787895, -0.015933630988001823,...","[0.1591796875, -0.0301513671875, 0.083984375, ...","[0.035617224872112274, -0.03864467144012451, 0...","[-0.009628409519791603, 0.027504952624440193, ...","[0.025325952097773552, -0.01553573738783598, -..."
3,4,Orbitalsysteme,GPT OSS,4,**Zusammenfassung** Die Raumfahrt des 21. Jahr...,"[-0.0075814262963831425, -0.015724439173936844...","[0.162109375, -0.03564453125, 0.08251953125, -...","[0.029788248240947723, -0.03914157301187515, 0...","[-0.01631634682416916, 0.033490538597106934, -...","[0.021283337846398354, -0.002345530316233635, ..."
4,5,Orbitalsysteme,Llama 3.1,1,### Zusammenfassung Die Raumfahrt des 21. Jahr...,"[-0.005312813911587, -0.013433435931801796, 0....","[0.1513671875, -0.043212890625, 0.08447265625,...","[0.0347779281437397, -0.04965607821941376, 0.0...","[0.010904254391789436, 0.02505870722234249, -0...","[0.016473066061735153, -0.0007392668048851192,..."



📄 Datei: combined_llm_embeddings.parquet
➡️ Anzahl Zeilen: 32
➡️ Spalten: ['experiment_id', 'audio_id', 'model_name', 'run_id', 'text', 'embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base']

🔍 df.head():


,experiment_id,audio_id,model_name,run_id,text,embedding_meta-llama_Llama-3.1-8B-Instruct,embedding_Jina_v3,embedding_AARI_German_Semantic_V3b,embedding_BGE_m3,embedding_Alibaba_gte_multilingual_base
0,1,Mythologien,GPT OSS,1,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.006361679174005985, -0.017968954518437386,...","[0.201171875, -0.12255859375, 0.1630859375, -0...","[0.04692911356687546, -0.07531177252531052, 0....","[-0.03595797345042229, 0.003199505852535367, -...","[-0.022312983870506287, 0.014333673752844334, ..."
1,2,Mythologien,GPT OSS,2,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.004676091484725475, -0.018816368654370308,...","[0.197265625, -0.11865234375, 0.1552734375, -0...","[0.04166073724627495, -0.07697830349206924, 0....","[-0.021111775189638138, 0.00964306015521288, -...","[-0.02722461149096489, 0.010483560152351856, -..."
2,3,Mythologien,GPT OSS,3,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.007357350084930658, -0.015717975795269012,...","[0.2021484375, -0.1181640625, 0.1494140625, -0...","[0.043072812259197235, -0.0732586681842804, 0....","[-0.016145789995789528, 0.010927277617156506, ...","[-0.0174740981310606, 0.00829455815255642, -0...."
3,4,Mythologien,GPT OSS,4,"**Zusammenfassung** Der Text beschreibt, dass ...","[-0.006447131745517254, -0.020988065749406815,...","[0.205078125, -0.10791015625, 0.1572265625, -0...","[0.04661232605576515, -0.06668359786272049, 0....","[-0.03934884071350098, 0.008379417471587658, 0...","[-0.02418340928852558, 0.005975430831313133, -..."
4,5,Mythologien,Llama 3.1,1,### Zusammenfassung: Die Moderne behauptet oft...,"[-0.009736663661897182, -0.014852538704872131,...","[0.17578125, -0.08251953125, 0.1337890625, -0....","[0.056146636605262756, -0.07163380086421967, 0...","[-0.00936897937208414, 0.01469916570931673, -0...","[-0.07917217165231705, 0.03245813772082329, -0..."



📄 Datei: transkripte_with_embeddings.parquet
➡️ Anzahl Zeilen: 2
➡️ Spalten: ['audio_id', 'text', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base', 'embedding_meta-llama_Llama-3.1-8B-Instruct']

🔍 df.head():


,audio_id,text,embedding_Jina_v3,embedding_AARI_German_Semantic_V3b,embedding_BGE_m3,embedding_Alibaba_gte_multilingual_base,embedding_meta-llama_Llama-3.1-8B-Instruct
0,Mythologien,Moderne Mythologien: Warum die Menschheit trot...,"[0.1884765625, -0.07958984375, 0.1435546875, -...","[0.04858698695898056, -0.06658167392015457, 0....","[-0.023830097168684006, -0.002423784462735057,...","[-0.02183479815721512, 0.007109793368726969, -...","[-0.0043811313807964325, -0.010360411368310452..."
1,Orbitalsysteme,Autonome Orbitalsysteme: Die stille technische...,"[0.166015625, -0.048828125, 0.0830078125, -0.0...","[0.03413056954741478, -0.04028322175145149, 0....","[-0.023131610825657845, 0.024594007059931755, ...","[0.0012604385847225785, 0.012034934014081955, ...","[-0.005769304931163788, 0.007544475607573986, ..."



✅ Alle Dateien erfolgreich angezeigt!


In [44]:
import pandas as pd

df_trans = pd.read_parquet("transkripte_with_embeddings.parquet")

rename_map = {
    "embedding_AARI_German_V3b": "embedding_AARI_German_Semantic_V3b",
    "embedding_Alibaba_GTE": "embedding_Alibaba_gte_multilingual_base"
}

df_trans = df_trans.rename(columns=rename_map)

df_trans.to_parquet("transkripte_with_embeddings.parquet")

print("Spalten erfolgreich angeglichen!")


import pandas as pd

# Datei laden
df = pd.read_parquet("combined_llm_embeddings.parquet")

# Kontrolle vorher
print("Vorher:", df["model_name"].unique())

# Ersetzen
df["model_name"] = df["model_name"].replace({"Llama": "Llama 3.1"})

# Kontrolle nachher
print("Nachher:", df["model_name"].unique())

# Speichern
df.to_parquet("combined_llm_embeddings.parquet")

print("\n✅ model_name 'Llama' → 'Llama 3.1' wurde erfolgreich ersetzt!")



import pandas as pd

file = "mythologien_with_embeddings.parquet"

# Datei laden
df = pd.read_parquet(file)

# Vorher prüfen
print("Vorher:", df["model_name"].unique())

# Ersetzen
df["model_name"] = df["model_name"].replace({"Llama": "Llama 3.1"})

# Nachher prüfen
print("Nachher:", df["model_name"].unique())

# Zurück speichern
df.to_parquet(file)

print(f"\n✅ 'Llama' → 'Llama 3.1' wurde erfolgreich geändert in: {file}")


Spalten erfolgreich angeglichen!
Vorher: ['GPT OSS' 'Llama 3.1' 'Medgemma' 'Sauerkraut']
Nachher: ['GPT OSS' 'Llama 3.1' 'Medgemma' 'Sauerkraut']

✅ model_name 'Llama' → 'Llama 3.1' wurde erfolgreich ersetzt!
Vorher: ['GPT OSS' 'Llama' 'Medgemma' 'Sauerkraut']
Nachher: ['GPT OSS' 'Llama 3.1' 'Medgemma' 'Sauerkraut']

✅ 'Llama' → 'Llama 3.1' wurde erfolgreich geändert in: mythologien_with_embeddings.parquet


## cosine_similarity Berechnung  für beide zusammen

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def cos_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return np.nan
    v1 = np.array(vec1).reshape(1, -1)
    v2 = np.array(vec2).reshape(1, -1)
    return float(cosine_similarity(v1, v2)[0][0])

# Dateien laden
df_llm = pd.read_parquet("combined_llm_embeddings.parquet")
df_trans = pd.read_parquet("transkripte_with_embeddings.parquet")

# KORREKTE Embedding-Spalten
embedding_cols = [
    "embedding_meta-llama_Llama-3.1-8B-Instruct",
    "embedding_Jina_v3",
    "embedding_AARI_German_Semantic_V3b",
    "embedding_BGE_m3",
    "embedding_Alibaba_gte_multilingual_base"
]

# Join
df_join = df_llm.merge(
    df_trans[["audio_id"] + embedding_cols],
    on="audio_id",
    suffixes=("_llm", "_trans")
)

# Similarity
for emb in embedding_cols:
    df_join[f"sim_{emb}"] = df_join.apply(
        lambda row: cos_sim(row[f"{emb}_llm"], row[f"{emb}_trans"]),
        axis=1
    )

# Aggregation pro Modell
final_scores = (
    df_join.groupby("model_name")[[f"sim_{emb}" for emb in embedding_cols]]
    .mean()
    .reset_index()
)

final_scores


,model_name,sim_embedding_meta-llama_Llama-3.1-8B-Instruct,sim_embedding_Jina_v3,sim_embedding_AARI_German_Semantic_V3b,sim_embedding_BGE_m3,sim_embedding_Alibaba_gte_multilingual_base
0,GPT OSS,0.477299,0.923054,0.989128,0.896267,0.885344
1,Llama 3.1,0.542411,0.914861,0.964799,0.875931,0.898056
2,Medgemma,0.618791,0.855033,0.979671,0.877103,0.870068
3,Sauerkraut,0.518029,0.892358,0.969040,0.875147,0.893132


In [4]:
# Alle Score-Spalten außer model_name
score_cols = [c for c in final_scores.columns if c != "model_name"]

sorted_dfs = {}

# Für jede Score-Spalte: Tabelle sortieren
for col in score_cols:
    sorted_dfs[col] = (
        final_scores[["model_name", col]]
        .sort_values(col, ascending=False)   # Höchster Score oben
        .reset_index(drop=True)
    )

# Alle sortierten Tabellen nebeneinander packen
combined = pd.concat(sorted_dfs.values(), axis=1)

combined


,model_name,sim_embedding_meta-llama_Llama-3.1-8B-Instruct,model_name,sim_embedding_Jina_v3,model_name,sim_embedding_AARI_German_Semantic_V3b,model_name,sim_embedding_BGE_m3,model_name,sim_embedding_Alibaba_gte_multilingual_base
0,Medgemma,0.618791,GPT OSS,0.923054,GPT OSS,0.989128,GPT OSS,0.896267,Llama 3.1,0.898056
1,Llama 3.1,0.542411,Llama 3.1,0.914861,Medgemma,0.979671,Medgemma,0.877103,Sauerkraut,0.893132
2,Sauerkraut,0.518029,Sauerkraut,0.892358,Sauerkraut,0.969040,Llama 3.1,0.875931,GPT OSS,0.885344
3,GPT OSS,0.477299,Medgemma,0.855033,Llama 3.1,0.964799,Sauerkraut,0.875147,Medgemma,0.870068


## für beide aber mit min max std

In [7]:
for emb in embedding_cols:
    sim_col = f"sim_{emb}"

    table = (
        df_join
        .groupby("model_name")[sim_col]
        .agg(
            mean="mean",
            min="min",
            max="max",
            std="std"
        )
        .reset_index()
        .sort_values("mean", ascending=False)  # 👈 wichtigste Zeile
    )

    print(f"\n=== Ergebnisse für {emb} (sortiert nach mean) ===")
    display(table)



=== Ergebnisse für embedding_meta-llama_Llama-3.1-8B-Instruct (sortiert nach mean) ===


,model_name,mean,min,max,std
2,Medgemma,0.618791,0.494778,0.881926,0.136208
1,Llama 3.1,0.542411,0.506690,0.569373,0.023389
3,Sauerkraut,0.518029,0.486189,0.550069,0.022723
0,GPT OSS,0.477299,0.423736,0.529000,0.031543



=== Ergebnisse für embedding_Jina_v3 (sortiert nach mean) ===


,model_name,mean,min,max,std
0,GPT OSS,0.923054,0.895214,0.945237,0.016990
1,Llama 3.1,0.914861,0.891021,0.951213,0.023453
3,Sauerkraut,0.892358,0.849229,0.930968,0.030227
2,Medgemma,0.855033,0.823910,0.883561,0.027554



=== Ergebnisse für embedding_AARI_German_Semantic_V3b (sortiert nach mean) ===


,model_name,mean,min,max,std
0,GPT OSS,0.989128,0.985989,0.993411,0.002408
2,Medgemma,0.979671,0.976097,0.983861,0.003073
3,Sauerkraut,0.969040,0.956387,0.976210,0.006858
1,Llama 3.1,0.964799,0.954603,0.977713,0.010136



=== Ergebnisse für embedding_BGE_m3 (sortiert nach mean) ===


,model_name,mean,min,max,std
0,GPT OSS,0.896267,0.879583,0.915443,0.011543
2,Medgemma,0.877103,0.843214,0.897911,0.023000
1,Llama 3.1,0.875931,0.862083,0.888934,0.008433
3,Sauerkraut,0.875147,0.857574,0.890987,0.011407



=== Ergebnisse für embedding_Alibaba_gte_multilingual_base (sortiert nach mean) ===


,model_name,mean,min,max,std
1,Llama 3.1,0.898056,0.848414,0.947452,0.047682
3,Sauerkraut,0.893132,0.842323,0.942337,0.038332
0,GPT OSS,0.885344,0.847949,0.918214,0.025432
2,Medgemma,0.870068,0.833840,0.905988,0.031653


## cosine_similarity Berechnung  für nur Mythologien

In [47]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------------------------------
# Cosine Similarity
# -------------------------------------------------------
def cos_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return np.nan
    v1 = np.array(vec1).reshape(1, -1)
    v2 = np.array(vec2).reshape(1, -1)
    return float(cosine_similarity(v1, v2)[0][0])


# -------------------------------------------------------
# 1️⃣ Dateien laden
# -------------------------------------------------------
df_llm = pd.read_parquet("mythologien_with_embeddings.parquet")
df_trans_full = pd.read_parquet("transkripte_with_embeddings.parquet")

# Nur das Mythologie-Transkript extrahieren
df_trans = df_trans_full[df_trans_full["audio_id"] == "Mythologien"].copy()

print("LLM Mythologien:", df_llm.shape)
print("Transkript Mythologien:", df_trans.shape)


# -------------------------------------------------------
# 2️⃣ Embedding-Spalten definieren (EXAKT wie in deinen Dateien)
# -------------------------------------------------------
embedding_cols = [
    "embedding_meta-llama_Llama-3.1-8B-Instruct",
    "embedding_Jina_v3",
    "embedding_AARI_German_Semantic_V3b",
    "embedding_BGE_m3",
    "embedding_Alibaba_gte_multilingual_base"
]


# -------------------------------------------------------
# 3️⃣ Join über audio_id
# -------------------------------------------------------
df_join = df_llm.merge(
    df_trans[["audio_id"] + embedding_cols],
    on="audio_id",
    suffixes=("_llm", "_trans")
)

print("Join-Resultat:", df_join.shape)


# -------------------------------------------------------
# 4️⃣ Cosine Similarity berechnen
# -------------------------------------------------------
for emb in embedding_cols:
    df_join[f"sim_{emb}"] = df_join.apply(
        lambda row: cos_sim(row[f"{emb}_llm"], row[f"{emb}_trans"]),
        axis=1
    )


# -------------------------------------------------------
# 5️⃣ Aggregation nach LLM-Modell
# -------------------------------------------------------
results = {}

for emb in embedding_cols:
    sim_col = f"sim_{emb}"
    results[emb] = (
        df_join.groupby("model_name")[sim_col]
        .agg(['mean', 'std', 'min', 'max', 'count'])
        .reset_index()
    )


# -------------------------------------------------------
# 6️⃣ Finale Score Tabelle erstellen
# -------------------------------------------------------
final_scores_myth = pd.DataFrame({"model_name": df_join["model_name"].unique()})

for emb in embedding_cols:
    final_scores_myth = final_scores_myth.merge(
        results[emb][["model_name", "mean"]].rename(columns={"mean": f"score_{emb}"}),
        on="model_name",
        how="left"
    )

final_scores_myth


LLM Mythologien: (16, 10)
Transkript Mythologien: (1, 7)
Join-Resultat: (16, 15)


,model_name,score_embedding_meta-llama_Llama-3.1-8B-Instruct,score_embedding_Jina_v3,score_embedding_AARI_German_Semantic_V3b,score_embedding_BGE_m3,score_embedding_Alibaba_gte_multilingual_base
0,GPT OSS,0.495098,0.909101,0.988548,0.902480,0.865068
1,Llama 3.1,0.562413,0.895604,0.955927,0.869778,0.853552
2,Medgemma,0.576276,0.880372,0.982415,0.896279,0.840985
3,Sauerkraut,0.531267,0.872214,0.965831,0.884693,0.867069


In [48]:
def build_ranking_matrix(final_scores):
    # Alle Score-Spalten außer model_name
    score_cols = [c for c in final_scores.columns if c != "model_name"]
    
    sorted_dfs = {}

    # Sortiert jede Spalte separat
    for col in score_cols:
        sorted_dfs[col] = (
            final_scores[["model_name", col]]
            .sort_values(col, ascending=False)  # höchster Score oben
            .reset_index(drop=True)
        )

    # Alle Rankings nebeneinander kleben
    combined = pd.concat(sorted_dfs.values(), axis=1)

    return combined


# -----------------------------------------------
# 📌  Erzeuge Ranking für Mythologien
# -----------------------------------------------
ranking_myth = build_ranking_matrix(final_scores_myth)
print("🎭 Ranking – Mythologien")
ranking_myth





🎭 Ranking – Mythologien


,model_name,score_embedding_meta-llama_Llama-3.1-8B-Instruct,model_name,score_embedding_Jina_v3,model_name,score_embedding_AARI_German_Semantic_V3b,model_name,score_embedding_BGE_m3,model_name,score_embedding_Alibaba_gte_multilingual_base
0,Medgemma,0.576276,GPT OSS,0.909101,GPT OSS,0.988548,GPT OSS,0.902480,Sauerkraut,0.867069
1,Llama 3.1,0.562413,Llama 3.1,0.895604,Medgemma,0.982415,Medgemma,0.896279,GPT OSS,0.865068
2,Sauerkraut,0.531267,Medgemma,0.880372,Sauerkraut,0.965831,Sauerkraut,0.884693,Llama 3.1,0.853552
3,GPT OSS,0.495098,Sauerkraut,0.872214,Llama 3.1,0.955927,Llama 3.1,0.869778,Medgemma,0.840985


## cosine_similarity Berechnung  für nur Orbirtalsysteme

In [49]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------------------------------
# Cosine Similarity
# -------------------------------------------------------
def cos_sim(vec1, vec2):
    if vec1 is None or vec2 is None:
        return np.nan
    v1 = np.array(vec1).reshape(1, -1)
    v2 = np.array(vec2).reshape(1, -1)
    return float(cosine_similarity(v1, v2)[0][0])


# -------------------------------------------------------
# 1️⃣ Dateien laden
# -------------------------------------------------------
df_llm = pd.read_parquet("orbitalsysteme_with_embeddings.parquet")
df_trans_full = pd.read_parquet("transkripte_with_embeddings.parquet")

# Nur das Orbitalsysteme-Transkript extrahieren
df_trans = df_trans_full[df_trans_full["audio_id"] == "Orbitalsysteme"].copy()

print("LLM Orbitalsysteme:", df_llm.shape)
print("Transkript Orbitalsysteme:", df_trans.shape)


# -------------------------------------------------------
# 2️⃣ Embedding-Spalten definieren (EXAKT wie in deinen Dateien)
# -------------------------------------------------------
embedding_cols = [
    "embedding_meta-llama_Llama-3.1-8B-Instruct",
    "embedding_Jina_v3",
    "embedding_AARI_German_Semantic_V3b",
    "embedding_BGE_m3",
    "embedding_Alibaba_gte_multilingual_base"
]


# -------------------------------------------------------
# 3️⃣ Join über audio_id
# -------------------------------------------------------
df_join = df_llm.merge(
    df_trans[["audio_id"] + embedding_cols],
    on="audio_id",
    suffixes=("_llm", "_trans")
)

print("Join-Resultat:", df_join.shape)


# -------------------------------------------------------
# 4️⃣ Cosine Similarity berechnen
# -------------------------------------------------------
for emb in embedding_cols:
    df_join[f"sim_{emb}"] = df_join.apply(
        lambda row: cos_sim(row[f"{emb}_llm"], row[f"{emb}_trans"]),
        axis=1
    )


# -------------------------------------------------------
# 5️⃣ Aggregation pro LLM-Modell
# -------------------------------------------------------
results = {}

for emb in embedding_cols:
    sim_col = f"sim_{emb}"
    results[emb] = (
        df_join.groupby("model_name")[sim_col]
        .agg(['mean', 'std', 'min', 'max', 'count'])
        .reset_index()
    )


# -------------------------------------------------------
# 6️⃣ Finale Score Tabelle erstellen
# -------------------------------------------------------
final_scores_orbit = pd.DataFrame({"model_name": df_join["model_name"].unique()})

for emb in embedding_cols:
    final_scores_orbit = final_scores_orbit.merge(
        results[emb][["model_name", "mean"]].rename(columns={"mean": f"score_{emb}"}),
        on="model_name",
        how="left"
    )

final_scores_orbit


LLM Orbitalsysteme: (16, 10)
Transkript Orbitalsysteme: (1, 7)
Join-Resultat: (16, 15)


,model_name,score_embedding_meta-llama_Llama-3.1-8B-Instruct,score_embedding_Jina_v3,score_embedding_AARI_German_Semantic_V3b,score_embedding_BGE_m3,score_embedding_Alibaba_gte_multilingual_base
0,GPT OSS,0.459500,0.937039,0.989707,0.890054,0.905619
1,Llama 3.1,0.522410,0.933924,0.973671,0.882085,0.942561
2,Medgemma,0.661306,0.829694,0.976928,0.857926,0.899150
3,Sauerkraut,0.504791,0.912540,0.972248,0.865600,0.919196


In [56]:
# -----------------------------------------------
# 📌  Erzeuge Ranking für Mythologien
# -----------------------------------------------
ranking_Orbirtalsysteme= build_ranking_matrix(final_scores_orbit)
print("🎭 Ranking – Orbirtalsysteme")
ranking_Orbirtalsysteme

🎭 Ranking – Orbirtalsysteme


,model_name,score_embedding_meta-llama_Llama-3.1-8B-Instruct,model_name,score_embedding_Jina_v3,model_name,score_embedding_AARI_German_Semantic_V3b,model_name,score_embedding_BGE_m3,model_name,score_embedding_Alibaba_gte_multilingual_base
0,Medgemma,0.661306,GPT OSS,0.937039,GPT OSS,0.989707,GPT OSS,0.890054,Llama 3.1,0.942561
1,Llama 3.1,0.522410,Llama 3.1,0.933924,Medgemma,0.976928,Llama 3.1,0.882085,Sauerkraut,0.919196
2,Sauerkraut,0.504791,Sauerkraut,0.912540,Llama 3.1,0.973671,Sauerkraut,0.865600,GPT OSS,0.905619
3,GPT OSS,0.459500,Medgemma,0.829694,Sauerkraut,0.972248,Medgemma,0.857926,Medgemma,0.899150


### Konsistenz innerhalb jedes Modelles 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import itertools

# -----------------------------
# 1️⃣ Datei laden
# -----------------------------
df = pd.read_parquet("combined_llm_embeddings.parquet")

# Alle Embedding-Spalten automatisch finden
embedding_cols = [c for c in df.columns if c.startswith("embedding_")]

print("Gefundene Embeddings:", embedding_cols)


# -----------------------------
# 2️⃣ Konsistenzfunktion für EIN Embedding
# -----------------------------
def compute_consistency(df, emb_col):
    consistency_results = []

    for model in df["model_name"].unique():
        model_subset = df[df["model_name"] == model]
        audio_scores = []

        for audio_id, subset in model_subset.groupby("audio_id"):
            if len(subset) < 2:
                continue

            embeddings = [
                np.array(e).reshape(1, -1)
                for e in subset[emb_col]
                if e is not None
            ]

            if len(embeddings) < 2:
                continue

            pairs = itertools.combinations(embeddings, 2)
            sims = [cosine_similarity(a, b)[0, 0] for a, b in pairs]

            audio_scores.append(np.mean(sims))

        if len(audio_scores) > 0:
            consistency_results.append({
                "model_name": model,
                "mean_consistency": float(np.mean(audio_scores))
            })

    return pd.DataFrame(consistency_results).sort_values(
        "mean_consistency", ascending=False
    )


# -----------------------------
# 3️⃣ Für jedes Embedding ein separates DataFrame berechnen
# -----------------------------
consistency_dfs = {}

for emb in embedding_cols:
    print(f"\n🔍 Berechne Konsistenz für Embedding: {emb}")
    df_emb = compute_consistency(df, emb)
    consistency_dfs[emb] = df_emb
    print(df_emb)   # Direktes Ausgeben des einzelnen DataFrames


Gefundene Embeddings: ['embedding_meta-llama_Llama-3.1-8B-Instruct', 'embedding_Jina_v3', 'embedding_AARI_German_Semantic_V3b', 'embedding_BGE_m3', 'embedding_Alibaba_gte_multilingual_base']

🔍 Berechne Konsistenz für Embedding: embedding_meta-llama_Llama-3.1-8B-Instruct
   model_name  mean_consistency
1   Llama 3.1          0.982266
0     GPT OSS          0.944903
2    Medgemma          0.879043
3  Sauerkraut          0.862515

🔍 Berechne Konsistenz für Embedding: embedding_Jina_v3
   model_name  mean_consistency
2    Medgemma          0.995127
0     GPT OSS          0.979980
1   Llama 3.1          0.977671
3  Sauerkraut          0.944727

🔍 Berechne Konsistenz für Embedding: embedding_AARI_German_Semantic_V3b
   model_name  mean_consistency
2    Medgemma          0.995258
0     GPT OSS          0.993769
1   Llama 3.1          0.991176
3  Sauerkraut          0.983280

🔍 Berechne Konsistenz für Embedding: embedding_BGE_m3
   model_name  mean_consistency
2    Medgemma          0.977990


: 

In [9]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import pandas as pd

# ============================================================
# 1. Original-DF laden
# ============================================================
df = pd.read_parquet("combined_llm_embeddings.parquet")

# ============================================================
# 2. Tokenizer laden (Jina v3)
# ============================================================
st_model = SentenceTransformer(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True
)
tokenizer = st_model[0].tokenizer

# ============================================================
# 3. Token pro Text berechnen
# ============================================================
rows = []

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Token zählen"):
    text = row.text if pd.notna(row.text) else ""

    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=True
    )

    token_count = tokens["input_ids"].shape[1]

    rows.append({
        "audio_id": row.audio_id,
        "model_name": row.model_name,
        "run_id": row.run_id,
        "token_count": token_count,
        "char_length": len(text),
        "text": text
    })

# ============================================================
# 4. Token-DF erstellen & speichern
# ============================================================
token_df = pd.DataFrame(rows)
token_df.to_parquet("token_counts_jina_v3.parquet")

print("\nToken-DF Vorschau:")
print(token_df.head())

# ============================================================
# 5. Statistik pro LLM-Modell
# ============================================================
stats = (
    token_df
    .groupby("model_name")
    .agg(
        mean_token_count=("token_count", "mean"),
        min_token_count=("token_count", "min"),
        max_token_count=("token_count", "max"),
        runs=("token_count", "count")
    )
    .sort_values("mean_token_count", ascending=False)
)

print("\nToken-Statistik pro Modell:")
print(stats)

# ============================================================
# 6. Optional: Statistik speichern
# ============================================================
stats.to_parquet("token_stats_by_model_jina_v3.parquet")


flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i


Token-DF Vorschau:
      audio_id model_name  run_id  token_count  char_length  \
0  Mythologien    GPT OSS       1         1084         4566   
1  Mythologien    GPT OSS       2         1199         5185   
2  Mythologien    GPT OSS       3         1111         4669   
3  Mythologien    GPT OSS       4          963         4104   
4  Mythologien  Llama 3.1       1          443         2041   

                                                text  
0  **Zusammenfassung** Der Text beschreibt, dass ...  
1  **Zusammenfassung** Der Text beschreibt, dass ...  
2  **Zusammenfassung** Der Text beschreibt, dass ...  
3  **Zusammenfassung** Der Text beschreibt, dass ...  
4  ### Zusammenfassung: Die Moderne behauptet oft...  

Token-Statistik pro Modell:
            mean_token_count  min_token_count  max_token_count  runs
model_name                                                          
GPT OSS             1148.000              963             1326     8
Medgemma             658.750       